# 🌲 Özellik Keşfi — Random Forest ile Özellik Önemi

Bu notebook, 4 hedef değişkenin her biri için **Random Forest** modeli eğiterek  
hangi giriş özelliklerinin tahmin gücü açısından en önemli olduğunu belirler.

**Random Forest özellik önemi:** Her ağaçta özelliğin sapma azaltmaya katkısı ölçülür.  
Bu sayede derin öğrenme modelini yorumlamak yerine, daha yorumlanabilir bir RF modeliyle  
özellik-hedef ilişkisi incelenebilir.

**Hedef değişkenler:**
1. `formation_energy_per_atom` — Formasyon enerjisi (eV/atom)
2. `band_gap` — Bant aralığı (eV)
3. `cbm` — İletim bandı minimum enerjisi (eV)
4. `energy_above_hull` — Hull üstü enerji / kararlılık göstergesi (eV/atom)

**Kullanılan dosyalar:** `data/X_preprocessed.csv`, `data/y_preprocessed.csv`

In [ ]:
# ============================================================
# KÜTÜPHANELER VE VERİ YÜKLEME
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

# Hazırlanmış özellik matrisi ve hedef matrisi yükle
# Bu dosyalar preparation.py çalıştırıldıktan sonra oluşur.
X = pd.read_csv("data/X_preprocessed.csv")
y = pd.read_csv("data/y_preprocessed.csv")

print(f"X boyutu: {X.shape}  (satır: örnek sayısı, sütun: özellik sayısı)")
print(f"y boyutu: {y.shape}  (sütunlar: {y.columns.tolist()})")

In [ ]:
# ============================================================
# YARDIMCI FONKSİYON — RF Eğit ve Önem Grafiği Çiz
# ============================================================
# Her hedef için aynı kodu tekrarlamamak adına bir fonksiyon yazıyoruz.

def rf_ozellik_onemi(X, y_series, hedef_adi, renk='steelblue', top_n=20):
    """
    Verilen X ve y ile Random Forest eğitir,
    test seti üzerinde MAE ve R² hesaplar,
    ardından en önemli top_n özelliği çubuk grafikte gösterir.

    Parametreler:
        X        : özellik matrisi (DataFrame)
        y_series : hedef vektörü (Series)
        hedef_adi: grafik başlığı için hedef adı (str)
        renk     : çubuk rengi (str)
        top_n    : kaç özelliği göster (int)
    """
    # Eğitim-test ayrımı: %80 eğitim, %20 test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_series, test_size=0.20, random_state=42
    )

    # Random Forest — n_jobs=-1 ile tüm CPU çekirdeklerini kullan
    rf = RandomForestRegressor(
        n_estimators=200,    # ağaç sayısı
        max_depth=15,        # derinlik sınırı — aşırı öğrenmeyi azaltır
        min_samples_leaf=4,  # yaprak başına minimum örnek
        n_jobs=-1,           # paralel hesaplama
        random_state=42
    )
    rf.fit(X_train, y_train)

    # Test seti metrikleri
    y_pred = rf.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    print(f"[{hedef_adi}]  MAE = {mae:.4f}   R² = {r2:.4f}")

    # Özellik önem sıralaması
    importances = pd.Series(rf.feature_importances_, index=X.columns)
    top = importances.sort_values(ascending=False).head(top_n)

    # Grafik
    fig, ax = plt.subplots(figsize=(10, 6))
    top[::-1].plot(kind='barh', ax=ax, color=renk, edgecolor='none', alpha=0.85)
    ax.set_xlabel('Özellik Önemi (Ortalama Safsızlık Azalması)', fontsize=11)
    ax.set_title(
        f'En Önemli {top_n} Özellik — {hedef_adi}\n'
        f'(RF: MAE={mae:.4f}, R²={r2:.4f})',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(f'figures/rf_onem_{hedef_adi.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    return importances

print("Yardımcı fonksiyon tanımlandı.")

In [ ]:
# ============================================================
# HEDEF 1: FORMASYON ENERJİSİ — formation_energy_per_atom
# ============================================================
# Formasyon enerjisi, bir kristalin oluşması için gereken enerjiyi gösterir.
# Negatif değerler termodinamik olarak elverişli (oluşabilir) malzemeleri işaret eder.

imp_fe = rf_ozellik_onemi(
    X, y['formation_energy_per_atom'],
    hedef_adi='Formasyon Enerjisi (eV/atom)',
    renk='#1f77b4'
)

In [ ]:
# ============================================================
# HEDEF 2: BANT ARALIĞI — band_gap
# ============================================================
# Bant aralığı, iletim bandı ile valans bandı arasındaki enerji farkıdır.
# band_gap = 0 → metal/iletken
# band_gap > 0 → yarı iletken veya yalıtkan
# NOT: GGA DFT verileri bant aralığını genellikle küçük tahmin eder.

imp_bg = rf_ozellik_onemi(
    X, y['band_gap'],
    hedef_adi='Bant Aralığı (eV)',
    renk='#ff7f0e'
)

In [ ]:
# ============================================================
# HEDEF 3: İLETİM BANDI MİNİMUMU — cbm
# ============================================================
# CBM (Conduction Band Minimum), iletim bandının en alt enerji seviyesidir.
# Negatif elektron ilgisi olan malzemelerde CBM negatif olabilir.
# band_gap ile güçlü korelasyon beklenir (band_gap = CBM - VBM).

imp_cbm = rf_ozellik_onemi(
    X, y['cbm'],
    hedef_adi='CBM (eV)',
    renk='#2ca02c'
)

In [ ]:
# ============================================================
# HEDEF 4: HULL ÜSTÜ ENERJİ — energy_above_hull
# ============================================================
# Hull üstü enerji, bir malzemenin denge faz diyagramına göre ne kadar
# kararsız olduğunu gösterir.
# ≤ 0.025 eV/atom → Stable   (kararlı, sentezlenebilir)
# ≤ 0.100 eV/atom → Metastable (koşullu kararlı)
# > 0.100 eV/atom → Unstable  (kararsız)

imp_hull = rf_ozellik_onemi(
    X, y['energy_above_hull'],
    hedef_adi='Hull Üstü Enerji (eV/atom)',
    renk='#d62728'
)

In [ ]:
# ============================================================
# ÖZET — 4 HEDEF İÇİN ÖNEMLİ ÖZELLİKLERİ KARŞILAŞTIR
# ============================================================
# Hangi özellikler birden fazla hedef için önemlidir?
# Bu, modelin farklı özelliklerden ortak bilgi çıkarabildiğini gösterir.

# Her hedef için top-10 özelliği tablo halinde göster
sonuclar = pd.DataFrame({
    'Formation Energy': imp_fe.sort_values(ascending=False).head(10).index,
    'Band Gap':         imp_bg.sort_values(ascending=False).head(10).index,
    'CBM':              imp_cbm.sort_values(ascending=False).head(10).index,
    'Energy Hull':      imp_hull.sort_values(ascending=False).head(10).index,
})

print("═" * 70)
print("Her hedef için en önemli 10 özellik:")
print("═" * 70)
print(sonuclar.to_string(index=True))

# Tüm hedeflerde ortak olan önemli özellikler
ortak = set(sonuclar['Formation Energy']) \
      & set(sonuclar['Band Gap']) \
      & set(sonuclar['CBM']) \
      & set(sonuclar['Energy Hull'])

print(f"\nTüm 4 hedef için ortak önemli özellikler ({len(ortak)} adet):")
for f in sorted(ortak):
    print(f"  • {f}")

## Özet ve Yorumlar

**Bant Aralığı (band_gap) için:**  
Elektron ilgisi (ea_*) ve elektronegativite (en_*) özellikleri genellikle en önemli sıralarda yer alır.  
Bu mantıklıdır: yüksek elektronegativiteli elementler (F, O, Cl) bant aralığını artırır.

**Formasyon Enerjisi için:**  
Ağırlıklı ortalama atom kütlesi (avg_atomic_mass) ve element fraksiyonları ön plana çıkar.  
Ağır elementler (Ba, La, Cs) içeren bileşikler tipik olarak daha negatif formasyon enerjisine sahiptir.

**Hull Üstü Enerji için:**  
Uzay grubu one-hot özellikleri ve element fraksiyonları önemlidir.  
Kararlılık, kristal simetrisine güçlü bağımlıdır.

**DFT Sınırlılığı (Fe2O3 Örneği):**  
Geçiş metali oksitlerinde (özellikle Fe, Co, Ni içerenler) GGA DFT band_gap'i sistematik olarak düşük tahmin eder.  
Bu yüzden modelin band_gap tahmini bu tür malzemelerde sapma göstermesi beklenen bir durumdur.  
Daha doğru tahminler için GGA+U veya hibrit fonksiyonel verisi kullanılması gerekir.  
Detaylı hata analizi için `examine.ipynb` defterine bakınız.